# Automação de Busca de Vagas - Gupy

Projeto de automação com Selenium para busca e coleta de vagas no Gupy.

## 1. Configuração inicial

Importação de bibliotecas e configuração do navegador.

In [153]:

from selenium import webdriver
from selenium.webdriver.remote.webdriver import WebDriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import pandas as pd
from urllib.parse import quote



## 2. Busca automatizada

Abrir o site, preencher a busca e confirmar.

In [154]:
# criar o navegador
servico = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=servico)
driver.get("https://portal.gupy.io/job-search")
driver.maximize_window()


In [155]:

def buscar_vagas(driver: WebDriver, termo: str) -> list:
    """
    Realiza a raspagem de vagas no Portal Gupy para uma localidade específica.

    Navega pelas páginas de resultados coletando informações detalhadas de cada 
    vaga disponível e trata possíveis instabilidades de carregamento ou fim de paginação.

    Args:
        driver (WebDriver): Instância ativa do navegador controlada pelo Selenium.
        termo (str): O cargo ou palavra-chave que será pesquisado.

    Returns:
        list[dict]: Uma lista de dicionários, onde cada dicionário contém as 
        informações de uma vaga (Título, Empresa, Local, Modelo, Tipo, Data, Link).
        Retorna uma lista vazia se nenhuma vaga for encontrada ou houver timeout.
    """
    nome_vaga = quote(termo)

    nome_uf = "São Paulo"
    nome_uf = quote(nome_uf)

    nome_cidade = "São Paulo"
    nome_cidade = quote(nome_cidade)

    link_vaga = f"https://portal.gupy.io/job-search/term={nome_vaga}&state={nome_uf}&city[]={nome_cidade}"

    driver.get(link_vaga)

    lista_vagas = []

    while True:
        try:
        
            try:
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href*="/job/"]')))
            except TimeoutException:
                mensagem_erro = f"Para o cargo '{termo}' não foram encontradas vagas ou a página falhou."
                print(mensagem_erro)
                print(f"Aviso: Tempo limite atingido para o cargo {termo}.")
                return lista_vagas

            vagas = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/job/"]')

            for vaga in vagas:
                
                titulo = vaga.find_element(By.TAG_NAME, "h3").text

                empresas = vaga.find_elements(By.TAG_NAME, "p")

                for empresa in empresas:
                    if not empresa.text.startswith("Publicada em:"):
                        nome_empresa = empresa.text
                    else:
                        data_vaga_publicada = empresa.text

                local = vaga.find_elements(By.CSS_SELECTOR, 'span[data-testid="job-location"]')

                if len(local) == 0:
                    local_vaga = None
                else:
                    local_vaga = local[0].text

                modelos_trabalho = ["Presencial", "Híbrido", "Remoto"]

                tipos_vaga = ["Estágio", "Efetivo", "Associado", "Autônomo", "Temporário", "Pessoa Jurídica", "Trainee", "Sócio"]

                elementos_span = vaga.find_elements(By.TAG_NAME, "span")

                modelo_encontrado = None
                tipo_vaga_encontrada = None
                pcd_encontrado = None

                for el_span in elementos_span:
                    if el_span.text in modelos_trabalho:
                        modelo_encontrado = el_span.text

                    elif el_span.text in tipos_vaga:
                        tipo_vaga_encontrada = el_span.text

                    elif el_span.text == "Também p/ PcD":
                        pcd_encontrado = el_span.text
                        
                link = vaga.get_attribute("href")

                dic_vagas = {
                    "Cargo Buscado": termo,
                    "Titulo": titulo,
                    "Empresa": nome_empresa,
                    "Local": local_vaga,
                    "Modelo": modelo_encontrado,
                    "Tipo da Vaga": tipo_vaga_encontrada,
                    "Afirmativa para PcD": pcd_encontrado,
                    "Data": data_vaga_publicada,
                    "Link": link
                    }

                lista_vagas.append(dic_vagas)

            proxima_pagina = driver.find_element(By.CSS_SELECTOR, 'button[aria-label="Próxima página"]')

            if proxima_pagina.is_enabled():
                proxima_pagina.click()
                time.sleep(1)

            else:
                break

        except NoSuchElementException:
            print(f"Botão não encontrado. Fim das páginas para {termo}.")
            break           
            
    return lista_vagas


In [156]:
'''driver.find_element(By.CSS_SELECTOR, 'input[placeholder="Busque por uma vaga"]').send_keys("Estágio TI")
driver.find_element(By.CSS_SELECTOR, 'button[data-testid="search-button"]').click()'''


'driver.find_element(By.CSS_SELECTOR, \'input[placeholder="Busque por uma vaga"]\').send_keys("Estágio TI")\ndriver.find_element(By.CSS_SELECTOR, \'button[data-testid="search-button"]\').click()'

In [157]:
'''driver.find_element(By.CSS_SELECTOR, 'button[aria-label="Local de trabalho"]').click()
time.sleep(1)

driver.find_element(By.ID, "dropdown-location-state-input").send_keys("São Paulo")
time.sleep(3)

opcoes_estado = driver.find_elements(By.CSS_SELECTOR, '[role="option"]')

estado = "São Paulo (SP)"

for opcao in opcoes_estado:
    if opcao.text == estado:
        opcao.click()
        break

time.sleep(3)

driver.find_element(By.ID, "multi-value-dropdown-location-city-input").send_keys("São Paulo")
time.sleep(2)

opcoes_cidades = driver.find_elements(By.CSS_SELECTOR, '[role="option"]')

cidade = "São Paulo"

for opcao in opcoes_cidades:
    if opcao.text == cidade:
        opcao.click()
        break

time.sleep(2)

driver.find_element(By.ID, "multi-value-dropdown-location-city-input").click()

time.sleep(3)
driver.find_element(By.XPATH, '//button[normalize-space()="Aplicar"]').click()'''




'driver.find_element(By.CSS_SELECTOR, \'button[aria-label="Local de trabalho"]\').click()\ntime.sleep(1)\n\ndriver.find_element(By.ID, "dropdown-location-state-input").send_keys("São Paulo")\ntime.sleep(3)\n\nopcoes_estado = driver.find_elements(By.CSS_SELECTOR, \'[role="option"]\')\n\nestado = "São Paulo (SP)"\n\nfor opcao in opcoes_estado:\n    if opcao.text == estado:\n        opcao.click()\n        break\n\ntime.sleep(3)\n\ndriver.find_element(By.ID, "multi-value-dropdown-location-city-input").send_keys("São Paulo")\ntime.sleep(2)\n\nopcoes_cidades = driver.find_elements(By.CSS_SELECTOR, \'[role="option"]\')\n\ncidade = "São Paulo"\n\nfor opcao in opcoes_cidades:\n    if opcao.text == cidade:\n        opcao.click()\n        break\n\ntime.sleep(2)\n\ndriver.find_element(By.ID, "multi-value-dropdown-location-city-input").click()\n\ntime.sleep(3)\ndriver.find_element(By.XPATH, \'//button[normalize-space()="Aplicar"]\').click()'

In [158]:
'''driver.find_element(By.CSS_SELECTOR, 'button[aria-label="Modelo de trabalho"]').click()

driver.find_element(By.NAME, "on-site").click()

driver.find_element(By.NAME, "hybrid").click()

driver.find_element(By.NAME, "remote").click()

driver.find_element(By.XPATH, '//button[normalize-space()="Aplicar"]').click()'''

'driver.find_element(By.CSS_SELECTOR, \'button[aria-label="Modelo de trabalho"]\').click()\n\ndriver.find_element(By.NAME, "on-site").click()\n\ndriver.find_element(By.NAME, "hybrid").click()\n\ndriver.find_element(By.NAME, "remote").click()\n\ndriver.find_element(By.XPATH, \'//button[normalize-space()="Aplicar"]\').click()'

## 3. Extração dos dados da página de resultados

Identificar e extrair título, empresa, local e link de cada vaga.

In [159]:
'''lista_vagas = []

vagas = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/job/"]')

for vaga in vagas:
    
    titulo = vaga.find_element(By.TAG_NAME, "h3").text

    empresas = vaga.find_elements(By.TAG_NAME, "p")

    for empresa in empresas:
        if not empresa.text.startswith("Publicada em:"):
            nome_empresa = empresa.text
        else:
            data_vaga_publicada = empresa.text

    local = vaga.find_elements(By.CSS_SELECTOR, 'span[data-testid="job-location"]')

    if len(local) == 0:
        local_vaga = None
    else:
        local_vaga = local[0].text

    modelos_trabalho = ["Presencial", "Híbrido", "Remoto"]

    tipos_vaga = ["Estágio", "Efetivo", "Associado", "Autônomo", "Temporário", "Pessoa Jurídica", "Trainee", "Sócio"]

    elementos_span = vaga.find_elements(By.TAG_NAME, "span")

    modelo_encontrado = None
    tipo_vaga_encontrada = None
    pcd_encontrado = None

    for el_span in elementos_span:
        if el_span.text in modelos_trabalho:
            modelo_encontrado = el_span.text

        elif el_span.text in tipos_vaga:
            tipo_vaga_encontrada = el_span.text

        elif el_span.text == "Também p/ PcD":
            pcd_encontrado = el_span.text
            
    link = vaga.get_attribute("href")


    dic_vagas = {
        "Cargo Buscado": termo,
        "Titulo": titulo,
        "Empresa": nome_empresa,
        "Local": local_vaga,
        "Modelo": modelo_encontrado,
        "Tipo da Vaga": tipo_vaga_encontrada,
        "Afirmativa para PcD": pcd_encontrado,
        "Data": data_vaga_publicada,
        "Link": link
        }

    lista_vagas.append(dic_vagas)'''
    


'lista_vagas = []\n\nvagas = driver.find_elements(By.CSS_SELECTOR, \'a[href*="/job/"]\')\n\nfor vaga in vagas:\n\n    titulo = vaga.find_element(By.TAG_NAME, "h3").text\n\n    empresas = vaga.find_elements(By.TAG_NAME, "p")\n\n    for empresa in empresas:\n        if not empresa.text.startswith("Publicada em:"):\n            nome_empresa = empresa.text\n        else:\n            data_vaga_publicada = empresa.text\n\n    local = vaga.find_elements(By.CSS_SELECTOR, \'span[data-testid="job-location"]\')\n\n    if len(local) == 0:\n        local_vaga = None\n    else:\n        local_vaga = local[0].text\n\n    modelos_trabalho = ["Presencial", "Híbrido", "Remoto"]\n\n    tipos_vaga = ["Estágio", "Efetivo", "Associado", "Autônomo", "Temporário", "Pessoa Jurídica", "Trainee", "Sócio"]\n\n    elementos_span = vaga.find_elements(By.TAG_NAME, "span")\n\n    modelo_encontrado = None\n    tipo_vaga_encontrada = None\n    pcd_encontrado = None\n\n    for el_span in elementos_span:\n        if el_

In [160]:

cargos = ["Estágio TI", "Analista de Dados Júnior", "Desenvolvedor Júnior", "Analista de Automação Júnior", "Automaçao Júnior"]

vagas_encontradas = []

for cargo in cargos:
    vagas_encontradas.extend(buscar_vagas(driver, cargo))
    time.sleep(1)



Para o cargo 'Analista de Automação Júnior' não foram encontradas vagas ou a página falhou.
Aviso: Tempo limite atingido para o cargo Analista de Automação Júnior.
Para o cargo 'Automaçao Júnior' não foram encontradas vagas ou a página falhou.
Aviso: Tempo limite atingido para o cargo Automaçao Júnior.


## 4. Paginação

Navegar pelas próximas páginas de resultados, se necessário.

In [161]:
'''pagina = driver.find_element(By.CSS_SELECTOR, 'button[aria-label="Próxima página"]').is_enabled()

print(pagina)'''

'pagina = driver.find_element(By.CSS_SELECTOR, \'button[aria-label="Próxima página"]\').is_enabled()\n\nprint(pagina)'

## 5. Organização e exportação dos dados

Transformar os resultados em DataFrame e exportar para CSV.

In [162]:
tabela_vagas = pd.DataFrame(vagas_encontradas)

tabela_vagas["Data"] = tabela_vagas["Data"].str.replace("Publicada em:", "", regex=False).str.strip()

tabela_vagas["Data"] = pd.to_datetime(tabela_vagas["Data"], format='%d/%m/%Y')

colunas_nulos = [col for col in tabela_vagas.columns if col != "Data"]

for coluna in colunas_nulos:
    tabela_vagas[coluna] = tabela_vagas[coluna].fillna("Não informado") 

tabela_vagas.to_csv("reports/vagas_encontradas.csv", index=False, sep=";", encoding="utf-8-sig", date_format='%d/%m/%Y')



display(tabela_vagas)
driver.close()

,Cargo Buscado,Titulo,Empresa,Local,Modelo,Tipo da Vaga,Afirmativa para PcD,Data,Link
0,Estágio TI,Estágio em TI | São Paulo/SP,Geração Cyrela,São Paulo - SP,Híbrido,Estágio,Também p/ PcD,2026-08-10,https://geracaocyrela.gupy.io/job/eyJqb2JJZCI6...
1,Estágio TI,Estágio Universitário - Suporte TI,Trabalhe na Mills,São Paulo - SP,Presencial,Estágio,Também p/ PcD,2026-08-05,https://programadeestagiomills.gupy.io/job/eyJ...
2,Estágio TI,Estágio Nível Superior - TI - Segurança da Inf...,CSN - Companhia Siderúrgica Nacional,São Paulo - SP,Presencial,Estágio,Também p/ PcD,2026-07-20,https://csn.gupy.io/job/eyJqb2JJZCI6MTE2NDkzND...
3,Estágio TI,Estágio em Suporte de TI - HÍBRIDO,#sejaveriter,São Paulo - SP,Híbrido,Efetivo,Também p/ PcD,2026-06-29,https://verity.gupy.io/job/eyJqb2JJZCI6MTE1NTg...
4,Estágio TI,Estágio em TI,BANCO DAYCOVAL,São Paulo - SP,Presencial,Efetivo,Também p/ PcD,2026-06-12,https://bancodaycoval.gupy.io/job/eyJqb2JJZCI6...
5,Estágio TI,Banco de Talentos Estágio em Desenvolvimento - TI,Gertec Brasil,Não informado,Não informado,Não informado,Não informado,2024-02-15,https://gertec.gupy.io/job/eyJqb2JJZCI6NjcyMzc...
6,Analista de Dados Júnior,Analista de Dados Júnior (Investigação de Frau...,Tokio Marine Seguradora,São Paulo - SP,Híbrido,Efetivo,Também p/ PcD,2026-08-10,https://tokiomarine.gupy.io/job/eyJqb2JJZCI6MT...
7,Analista de Dados Júnior,Analista de Dados Júnior (Precificação e Intel...,Tokio Marine Seguradora,São Paulo - SP,Híbrido,Efetivo,Também p/ PcD,2026-08-10,https://tokiomarine.gupy.io/job/eyJqb2JJZCI6MT...
8,Analista de Dados Júnior,Analista de Dados SRO Júnior,Tokio Marine Seguradora,São Paulo - SP,Híbrido,Efetivo,Também p/ PcD,2026-07-31,https://tokiomarine.gupy.io/job/eyJqb2JJZCI6MT...
9,Analista de Dados Júnior,ANALISTA DE DADOS JUNIOR | PREVENÇÃO E PERDAS,Riachuelo,São Paulo - SP,Híbrido,Efetivo,Também p/ PcD,2026-07-30,https://riachuelo.gupy.io/job/eyJqb2JJZCI6MTE4...
